In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans,DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from fancyimpute import IterativeImputer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from collections import defaultdict
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors

import os 
os.environ['OMP_NUMRHREADS']='1'

In [2]:
rent_train=pd.read_csv('data/ruc_Class25Q2_train_rent.csv')
rent_test=pd.read_csv('data/ruc_Class25Q2_test_rent.csv')

C:\Users\oasis\AppData\Local\Temp\ipykernel_11516\3437492410.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  rent_train=pd.read_csv('data/ruc_Class25Q2_train_rent.csv')


In [3]:
# 保存原始索引
rent_train_original_index = rent_train.index.copy()
rent_test_original_index = rent_test.index.copy()

#安全提取目标变量和ID
target_Price=rent_train['Price']
rent_train=rent_train.drop('Price',axis=1)
rent_train=rent_train.drop(['朝向','年份','物业类别','物业办公电话','coord_x','coord_y','客户反馈'], axis=1)
                             
rent_train.columns = rent_train.columns.str.strip()
rent_train.columns = [col.replace(' ', '') for col in rent_train.columns]

rent_test_id = rent_test['ID'].copy() 
rent_test=rent_test.drop('ID',axis=1)
rent_test=rent_test.drop(['朝向','年份','物业类别','物业办公电话','coord_x','coord_y','客户反馈'], axis=1)
                             
rent_test.columns = rent_test.columns.str.strip()
rent_test.columns = [col.replace(' ', '') for col in rent_test.columns]

In [4]:
rent_train.info()
rent_train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 38 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      98899 non-null  int64  
 1   户型      98898 non-null  object 
 2   装修      25410 non-null  object 
 3   楼层      98894 non-null  object 
 4   面积      98899 non-null  object 
 5   交易时间    98899 non-null  object 
 6   付款方式    80476 non-null  object 
 7   租赁方式    98899 non-null  object 
 8   电梯      98895 non-null  object 
 9   车位      24764 non-null  object 
 10  用水      81159 non-null  object 
 11  用电      81575 non-null  object 
 12  燃气      94317 non-null  object 
 13  采暖      34412 non-null  object 
 14  租期      51966 non-null  object 
 15  配套设施    68448 non-null  object 
 16  lon     98899 non-null  float64
 17  lat     98899 non-null  float64
 18  区县      94222 non-null  float64
 19  板块      93755 non-null  float64
 20  环线位置    29236 non-null  object 
 21  建筑年代    72750 non-null  object 
 22

,城市,lon,lat,区县,板块,容积率,停车位
count,98899.000000,98899.000000,98899.000000,94222.000000,93755.000000,74819.000000,73420.000000
mean,4.322157,115.752394,31.420651,70.243054,588.404096,3.060876,1285.232866
std,3.321254,5.578828,6.378794,35.832629,344.070325,1.751343,1576.732337
min,0.000000,103.482606,23.025849,3.000000,1.000000,0.020000,1.000000
25%,2.000000,114.357093,24.201337,38.000000,298.000000,2.000000,300.000000
50%,3.000000,117.156819,32.163991,70.000000,585.000000,2.670000,779.000000
75%,7.000000,121.644325,37.606197,95.000000,895.000000,3.800000,1590.000000
max,11.000000,122.966660,42.189642,131.000000,1186.000000,30.000000,8700.000000


In [5]:
rent_test.info()
rent_test.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9773 entries, 0 to 9772
Data columns (total 38 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      9773 non-null   int64  
 1   户型      9773 non-null   object 
 2   装修      7091 non-null   object 
 3   楼层      9773 non-null   object 
 4   面积      9773 non-null   object 
 5   交易时间    9773 non-null   object 
 6   付款方式    7386 non-null   object 
 7   租赁方式    9773 non-null   object 
 8   电梯      9773 non-null   object 
 9   车位      2162 non-null   object 
 10  用水      7500 non-null   object 
 11  用电      7546 non-null   object 
 12  燃气      9182 non-null   object 
 13  采暖      3213 non-null   object 
 14  租期      4598 non-null   object 
 15  配套设施    6079 non-null   object 
 16  lon     9773 non-null   float64
 17  lat     9773 non-null   float64
 18  区县      8848 non-null   float64
 19  板块      8837 non-null   float64
 20  环线位置    2811 non-null   object 
 21  建筑年代    6392 non-null   object 
 22  

,城市,lon,lat,区县,板块,容积率,停车位
count,9773.000000,9773.000000,9773.000000,8848.000000,8837.000000,6785.000000,6658.000000
mean,4.358027,116.198236,31.790427,70.205922,593.869413,2.945477,1229.518023
std,3.273741,5.432899,6.367373,35.249205,335.297322,1.671420,1538.576934
min,0.000000,103.482657,23.025980,3.000000,1.000000,0.020000,1.000000
25%,2.000000,114.443352,24.404857,42.000000,303.000000,1.900000,300.000000
50%,4.000000,117.322625,32.253961,68.000000,583.000000,2.500000,700.000000
75%,7.000000,121.773673,37.637167,95.000000,891.000000,3.500000,1500.000000
max,11.000000,122.966575,42.693635,131.000000,1186.000000,18.800000,8700.000000


In [6]:
def extract_fee_value(text):
    """提取费用数值并计算平均值（适用于物业费、燃气费、供热费等）"""
    if pd.isna(text):
        return None
    
    numbers = re.findall(r'\d+\.?\d*', str(text).replace(' ', ''))
    if numbers:
        return sum(map(float, numbers)) / len(numbers)
    return np.nan

def adjust_green_rate(x):
                    if pd.isna(x):
                        return x
                    # 如果大于100，尝试除以100
                    if x > 100:
                        x_adjusted = x / 100
                        # 如果调整后的值在0-100之间，则返回调整后的值，否则返回NaN
                        if 0 <= x_adjusted <= 100:
                            return x_adjusted
                        else:
                            return np.nan
                    # 如果小于0，调整为0
                    elif x < 0:
                        return 0.0
                    else:
                        return x
                        
def extract_num(df, columns=['面积','房屋总数','楼栋总数','物业费','燃气费','供热费','绿化率']):
    for col in columns:
        if col == '面积':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)㎡')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col == '房屋总数':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)户')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col == '楼栋总数':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)栋')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col in ['物业费', '燃气费', '供热费']:
            # 使用统一的提取函数
            df[col] = df[col].apply(extract_fee_value)
        elif col == '绿化率':
            df[col] = df[col].astype(str).str.extract(r'(\d+(?:\.\d+)?)\s*%?')
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].apply(adjust_green_rate)
            
    return df

rent_train=extract_num(rent_train)
rent_test=extract_num(rent_test)

In [7]:
def clean_floor_data(df, column_name='楼层'):
    """
    清洗楼层数据，统一转换为当前楼层/总楼层的比值
    """
    # 创建一个副本以避免修改原数据
    df_clean = df.copy()
    
    def clean_single_floor(floor_str):
        """
        清洗单个楼层字符串，返回楼层比值
        """
        # 处理空值
        if pd.isna(floor_str) or floor_str == '':
            return None
        
        floor_str = str(floor_str).strip()
        
        # 情况1: 已经是标准格式 "x/y层" - 提取数字并计算比值
        if re.match(r'^\d+/\d+层$', floor_str):
            numbers = re.findall(r'\d+', floor_str)
            if len(numbers) == 2:
                current, total = map(int, numbers)
                return current / total if total > 0 else None
        
        # 情况2: 有前缀描述 "中楼层/25层" - 估算当前楼层并计算比值
        elif '/' in floor_str and any(keyword in floor_str for keyword in ['低楼层', '中楼层', '高楼层']):
            parts = floor_str.split('/')
            if len(parts) == 2:
                # 提取总楼层数字
                total_match = re.search(r'(\d+)', parts[1])
                if total_match:
                    total_floor = int(total_match.group(1))
                    if total_floor <= 0:
                        return None
                    
                    # 根据描述估算当前楼层
                    if '低楼层' in parts[0]:
                        current = max(1, total_floor // 3)
                    elif '中楼层' in parts[0]:
                        current = max(1, total_floor // 2)
                    elif '高楼层' in parts[0]:
                        current = max(1, total_floor * 2 // 3)
                    else:
                        current = 1  # 默认
                    
                    return current / total_floor
        
        # 情况3: 只有数字格式 "4/6" - 直接计算比值
        elif re.match(r'^\d+/\d+$', floor_str):
            current, total = map(int, floor_str.split('/'))
            return current / total if total > 0 else None
        
        # 情况4: 其他格式尝试提取数字
        else:
            numbers = re.findall(r'\d+', floor_str)
            if len(numbers) >= 2:
                current, total = int(numbers[0]), int(numbers[1])
                return current / total if total > 0 else None
        
        return None
    
    # 应用清洗函数到指定列
    df_clean[column_name] = df_clean[column_name].apply(clean_single_floor)
    
    return df_clean

# 应用清洗函数
rent_train = clean_floor_data(rent_train)
rent_test = clean_floor_data(rent_test)

In [8]:
def convert_dummies(df):
    # 是否有电梯
    if '电梯' in df.columns:  
        df['电梯'] = df['电梯'].map({'有': 1, '无': 0})
        df['电梯'] = df['电梯'].fillna('0')

    # 是否有燃气
    if '燃气' in df.columns:  
        df['燃气'] = df['燃气'].map({'有': 1, '无': 0})
        df['燃气'] = df['燃气'].fillna('0')

    if '租赁方式' in df.columns:  
        df['租赁方式'] = df['租赁方式'].map({'整租': 1, '合租': 0})

    if '装修'in df.columns:
        df['装修'] = df['装修'].map({'精装修': 1})
        df['装修'] = df['装修'].fillna('0')

    return df

rent_train = convert_dummies(rent_train)
rent_test = convert_dummies(rent_test) 

In [9]:
def simple_extractnum(df):       
    # 房屋户型
    if "户型" in df.columns:
        def parse_layout(text):
            text = str(text)
            rooms = re.findall(r'(\d+)室', text)
            halls = re.findall(r'(\d+)厅', text)
            baths = re.findall(r'(\d+)卫', text)
            return pd.Series([
                int(rooms[0]) if rooms else 0,
                int(halls[0]) if halls else 0,
                int(baths[0]) if baths else 0
            ])
        df[["卧室数","客厅数","卫生间数"]] = df["户型"].apply(parse_layout)


    # 配套设施数量
    if '配套设施' in df.columns:
        def count_facilities(text):
            """
            计算配套设施数量
            """
            if pd.isna(text) or text == '':
                return 0
            
            text = str(text).strip()
            # 使用顿号分割设施
            facilities = [facility.strip() for facility in text.split('、') if facility.strip()]
            return len(facilities)
        
        # 添加配套设施数量列
        df['配套设施数量'] = df['配套设施'].apply(count_facilities)
        
        # 定义常见配套设施
        common_facilities = ['洗衣机', '空调', '衣柜', '电视', '冰箱', '热水器', '床', '暖气', '宽带', '天然气']
        
        def has_common_facilities(text, threshold=0.7):
            """
            检查是否有常见配套设施
            threshold: 阈值，当包含的常见设施比例超过此值时返回1
            """
            if pd.isna(text) or text == '':
                return 0
            
            text = str(text).strip()
            facilities_list = [facility.strip() for facility in text.split('、') if facility.strip()]
            
            # 计算包含的常见设施数量
            common_count = sum(1 for facility in common_facilities if facility in facilities_list)
            
            # 如果包含的常见设施比例超过阈值，则返回1
            return 1 if common_count / len(common_facilities) >= threshold and common_count>=6 else 0
        
        # 创建"设施是否完善"列
        df['设施是否完善'] = df['配套设施'].apply(has_common_facilities)

    return df


rent_train = simple_extractnum(rent_train)
rent_test = simple_extractnum(rent_test)

In [10]:
# 停车费用处理
# 1. 通用金额→月费换算函数
def convert_to_month_fee(value, unit_hint=None):
    """根据金额及上下文提示换算为月费（元/月）"""
    if value is None or pd.isna(value):
        return np.nan
    value = float(value)
    
    # 单位提示优先
    if unit_hint == 'hour':
        return value * 240
    elif unit_hint == 'day':
        return value * 30
    elif unit_hint == 'year':
        return value / 12
    elif unit_hint == 'month':
        return value
    
    # 没有单位提示 -> 通过数值范围推断
    if value == 0:
        return 0
    elif value < 20:        # 太小 -> 按小时算
        return value * 240
    elif value < 1000:     # 正常区间 -> 月费
        return value
    elif value >= 1000:    # 超大值 -> 售价，忽略
        return np.nan
    
    return np.nan
# 2 中文数字单位转化
def chinese_number_to_float(text):
    """
    将带有中文单位的数字（如 '1万', '2千', '3百'）转换为浮点数。
    不带单位的数字直接转 float。
    """
    if text is None or text == '':
        return np.nan

    text = str(text).strip()
    
    # 去除“元”、“块”、“/月”等干扰字符
    t = re.sub(r'[元块\/每月位]+', '', text)
    
    # 处理带单位的情况
    if re.search(r'万', t):
        base = float(re.sub(r'万.*', '', t))
        return base * 10000
    elif re.search(r'千', t):
        base = float(re.sub(r'千.*', '', t))
        return base * 1000
    elif re.search(r'百', t):
        base = float(re.sub(r'百.*', '', t))
        return base * 100
    else:
        # 无单位时，取第一个数字
        match = re.search(r'\d+\.?\d*', t)
        if match:
            return float(match.group())
        else:
            return np.nan
            
# 3 从文本中提取所有相关费用的函数
def extract_values(text):
    """解析原始文本，返回提取出的多种类型价格"""
    if not isinstance(text, str) or text.strip() == '':
        return {}
    
    t = text.replace('～','-').replace('~','-').replace('；',',').replace('。',',').replace('、',',')

    #将中文转化为数字
    for m in re.finditer(r'([\d\.]+(?:万|千|百)?)[元块]*/?(小时|时|天|月|年)?', t):
        val_raw, unit = m.groups()
        val = chinese_number_to_float(val_raw)
    
    # 免费
    if re.search(r'免费|不收费|0元', t):
        return {'free': True}
    
    result = {
        'ground': [], 'underground': [], 'outdoor': [], 'indoor': [],
        'fixed': [], 'unfixed': [],
        'owner': [], 'tenant': [],
        'hour': [], 'day': [], 'month': [], 'year': []
    }
    
    # 区间价（如300-500）
    ranges = re.findall(r'(\d+\.?\d*)[-~～](\d+\.?\d*)', t)
    for a, b in ranges:
        result['month'].append((float(a)+float(b))/2)
    
    # 单价识别（带单位）
    for m in re.finditer(r'(\d+\.?\d*)元?/?(小时|时|天|月|年)?', t):
        val, unit = m.groups()
        if not val:
            continue
        val = float(val)
        unit_map = {'小时':'hour','时':'hour','天':'day','月':'month','年':'year'}
        unit = unit_map.get(unit, None)
        
        # 归类主体
        segment = t[max(0, m.start()-6):m.end()+6]
        if re.search(r'地上', segment):
            result['ground'].append(convert_to_month_fee(val, unit))
        elif re.search(r'地下', segment):
            result['underground'].append(convert_to_month_fee(val, unit))
        elif re.search(r'露天', segment):
            result['outdoor'].append(convert_to_month_fee(val, unit))
        elif re.search(r'室内', segment):
            result['indoor'].append(convert_to_month_fee(val, unit))
        elif re.search(r'固定', segment) and not re.search(r'不固定', segment):
            result['fixed'].append(convert_to_month_fee(val, unit))
        elif re.search(r'不固定', segment):
            result['unfixed'].append(convert_to_month_fee(val, unit))
        elif re.search(r'业主', segment):
            result['owner'].append(convert_to_month_fee(val, unit))
        elif re.search(r'租客', segment):
            result['tenant'].append(convert_to_month_fee(val, unit))
        elif re.search(r'临保|小时|时', segment):
            result['hour'].append(convert_to_month_fee(val, unit))
        else:
            result['month'].append(convert_to_month_fee(val, unit))
    
    return result

# 3️ 主函数：从多类型结果中统一为一个月度费用
def unify_month_fee(text):
    info = extract_values(text)
    
    if info.get('free'):
        return 0.0
    
    # 地上/地下/露天/室内 → 环境类取均值
    env_fees = []
    for key in ['ground','underground','outdoor','indoor']:
        vals = [v for v in info.get(key, []) if v is not None and not pd.isna(v)]
        if len(vals)>0:
            env_fees.extend(vals)
    if len(env_fees)==1:
        env_fee = env_fees[0]
    elif len(env_fees)>=2:
        # 如果包含0，则取非0均值
        non_zero = [v for v in env_fees if v>0]
        env_fee = np.mean(non_zero) if len(non_zero)>0 else 0
    else:
        env_fee = np.nan
    
    # 固定/不固定取均值
    fix_vals = [v for v in info.get('fixed', []) + info.get('unfixed', []) if not pd.isna(v)]
    fix_fee = np.mean(fix_vals) if len(fix_vals)>0 else np.nan
    
    # 业主/租客取均值
    own_vals = [v for v in info.get('owner', []) + info.get('tenant', []) if not pd.isna(v)]
    own_fee = np.mean(own_vals) if len(own_vals)>0 else np.nan
    
    # 临保/小时取均值
    hour_vals = [v for v in info.get('hour', []) if not pd.isna(v)]
    hour_fee = np.mean(hour_vals) if len(hour_vals)>0 else np.nan
    
    # 普通月费
    month_vals = [v for v in info.get('month', []) if not pd.isna(v)]
    month_fee = np.mean(month_vals) if len(month_vals)>0 else np.nan
    
    # 综合选择优先级：环境类 > 固定类 > 业主类 > 月保 > 小时类
    candidates = [env_fee, fix_fee, own_fee, month_fee, hour_fee]
    candidates = [v for v in candidates if not pd.isna(v)]
    
    if len(candidates)==0:
        return np.nan
    else:
        return round(float(np.mean(candidates)), 2)

# 4️ 应用并输出结果

rent_train['停车费用'] = rent_train['停车费用'].apply(unify_month_fee)
rent_test['停车费用'] = rent_test['停车费用'].apply(unify_month_fee)

In [11]:
def extract_years(text):
    """从字符串中提取起始年份和结束年份"""
    years = re.findall(r'\d{4}', str(text))
    
    if not years:
        return np.nan, np.nan
    elif len(years) == 1:
        return int(years[0]), int(years[0])  # 单一年份
    else:
        return int(years[0]), int(years[1])  # 年份区间
def process_construction_period(df, column_name='建筑年代'):
    """处理建筑年代区间数据"""
    df = df.copy()

    # 提取年份并计算相关特征
    df[['建筑起始年份', '建筑结束年份']] = df[column_name].apply(
        lambda x: pd.Series(extract_years(x)) if not pd.isna(x) else pd.Series([np.nan, np.nan])
    )

    # 计算建筑特征
    df['平均建筑年份'] = (df['建筑起始年份'] + df['建筑结束年份']) / 2
    
    # 创建分组和质量特征
    df['交易时间'] = pd.to_datetime(df['交易时间'], errors='coerce')
    trade_year = df['交易时间'].dt.year
    
    # 房龄特征
    df['小区最新房龄'] = trade_year - df['建筑结束年份']
    df['小区最老房龄'] = trade_year - df['建筑起始年份']
    df['小区平均房龄'] = (df['小区最新房龄'] + df['小区最老房龄']) / 2
    df['小区房龄差异'] = df['小区最老房龄'] - df['小区最新房龄']
    
    # 政策周期编号
    policy_conditions = [
        df['建筑结束年份'] < 1998,
        (df['建筑起始年份'] >= 1998) & (df['建筑结束年份'] <= 2007),
        (df['建筑起始年份'] >= 2008) & (df['建筑结束年份'] <= 2010),
        (df['建筑起始年份'] >= 2011) & (df['建筑结束年份'] <= 2014),
        (df['建筑起始年份'] >= 2015) & (df['建筑结束年份'] <= 2016),
        df['建筑起始年份'] >= 2017
    ]
    
    policy_periods = ['房改前', '黄金十年', '四万亿', '限购期', '去库存', '房住不炒']

    df['政策周期'] = np.select(policy_conditions, policy_periods, default='未知')

    df = df.drop(['建筑起始年份', '建筑结束年份','小区最新房龄','小区最老房龄','平均建筑年份'], axis=1)
    return df

rent_train=process_construction_period(rent_train)
rent_test=process_construction_period(rent_test)

In [12]:
num_cols = rent_train.select_dtypes(include=[np.number]).columns.tolist()

In [13]:
class SimplifiedGeoClusterMapper:
    def __init__(self, grid_size=0.01, min_samples_per_grid=5):
        """
        简化版地理聚类 - 使用网格方法
        
        参数:
            grid_size: 网格大小（经纬度单位）
            min_samples_per_grid: 每个网格最少样本数
        """
        self.grid_size = grid_size
        self.min_samples_per_grid = min_samples_per_grid
        self.city_grid_models = {}
        self.is_fitted = False
    
    def fit(self, df):
        """拟合模型 - 简化的网格方法"""
        print("开始地理聚类...")
        results = []
        
        for city in df['城市'].unique():
            city_data = df[df['城市'] == city].copy()
            print(f"处理城市: {city}, 样本数: {len(city_data)}")
            
            # 为城市创建网格
            # 计算网格坐标
            city_data['grid_lon'] = (city_data['lon'] / self.grid_size).astype(int)
            city_data['grid_lat'] = (city_data['lat'] / self.grid_size).astype(int)
            
            # 创建网格ID
            city_data['grid_id'] = (
                city_data['grid_lon'].astype(str) + '_' + 
                city_data['grid_lat'].astype(str)
            )
            
            # 统计每个网格的样本数
            grid_counts = city_data['grid_id'].value_counts()
            
            # 为每个网格分配聚类ID
            grid_to_cluster = {}
            current_cluster_id = 0
            
            for grid_id, count in grid_counts.items():
                if count >= self.min_samples_per_grid:
                    grid_to_cluster[grid_id] = current_cluster_id
                    current_cluster_id += 1
                else:
                    # 样本少的网格标记为-1（将在后面处理）
                    grid_to_cluster[grid_id] = -1
            
            # 应用网格聚类
            city_data['地理聚类'] = city_data['grid_id'].map(grid_to_cluster)
            
            # 处理样本少的网格 - 合并到邻近网格或使用区县信息
            sparse_mask = city_data['地理聚类'] == -1
            if sparse_mask.sum() > 0:
                print(f"  {city}有{sparse_mask.sum()}个稀疏网格样本需要处理")
                
                # 方法1: 尝试合并到邻近的稠密网格
                sparse_data = city_data[sparse_mask].copy()
                for idx, row in sparse_data.iterrows():
                    # 查找邻近的稠密网格
                    nearby_grids = city_data[
                        (city_data['grid_lon'].between(row['grid_lon']-1, row['grid_lon']+1)) &
                        (city_data['grid_lat'].between(row['grid_lat']-1, row['grid_lat']+1)) &
                        (city_data['地理聚类'] != -1)
                    ]
                    
                    if len(nearby_grids) > 0:
                        # 使用邻近网格的聚类ID
                        city_data.loc[idx, '地理聚类'] = nearby_grids['地理聚类'].iloc[0]
                    else:
                        # 方法2: 使用区县信息
                        region_data = city_data[
                            (city_data['区县'] == row['区县']) & 
                            (city_data['地理聚类'] != -1)
                        ]
                        if len(region_data) > 0:
                            city_data.loc[idx, '地理聚类'] = region_data['地理聚类'].mode().iloc[0] if len(region_data['地理聚类'].mode()) > 0 else -1
            
            # 创建聚类层级（基于聚类大小）
            cluster_sizes = city_data['地理聚类'].value_counts()
            
            def assign_cluster_level(cluster_id):
                if cluster_id == -1:
                    return 0  # 未知层级
                size = cluster_sizes.get(cluster_id, 0)
                if size >= 100:
                    return 3  # 大型聚类
                elif size >= 20:
                    return 2  # 中型聚类
                elif size >= 5:
                    return 1  # 小型聚类
                else:
                    return 0  # 微型聚类
            
            city_data['聚类层级'] = city_data['地理聚类'].apply(assign_cluster_level)
            
            # 保存城市模型信息
            self.city_grid_models[city] = {
                'grid_to_cluster': grid_to_cluster,
                'cluster_sizes': cluster_sizes,
                'min_lon': city_data['grid_lon'].min(),
                'max_lon': city_data['grid_lon'].max(),
                'min_lat': city_data['grid_lat'].min(),
                'max_lat': city_data['grid_lat'].max()
            }
            
            results.append(city_data)
        
        # 合并所有城市的结果
        result_df = pd.concat(results, ignore_index=True)
        
        # 删除临时列
        result_df = result_df.drop(['grid_lon', 'grid_lat', 'grid_id'], axis=1)
        
        self.is_fitted = True
        print(f"地理聚类完成，总样本数: {len(result_df)}")
        print(f"聚类统计: {result_df['地理聚类'].nunique()} 个聚类")
        print(f"聚类层级分布: {result_df['聚类层级'].value_counts().to_dict()}")
        
        return result_df
    
    def transform(self, df):
        """对测试集应用聚类"""
        if not self.is_fitted:
            raise ValueError("必须先调用fit方法")
        
        print("对测试集应用地理聚类...")
        results = []
        
        for city in df['城市'].unique():
            city_data = df[df['城市'] == city].copy()
            
            if city not in self.city_grid_models:
                print(f"警告: 城市 {city} 在训练集中未出现，使用默认聚类")
                city_data['地理聚类'] = -1
                city_data['聚类层级'] = 0
                results.append(city_data)
                continue
            
            city_model = self.city_grid_models[city]
            
            # 计算网格坐标
            city_data['grid_lon'] = (city_data['lon'] / self.grid_size).astype(int)
            city_data['grid_lat'] = (city_data['lat'] / self.grid_size).astype(int)
            
            # 创建网格ID
            city_data['grid_id'] = (
                city_data['grid_lon'].astype(str) + '_' + 
                city_data['grid_lat'].astype(str)
            )
            
            # 应用聚类映射
            city_data['地理聚类'] = city_data['grid_id'].map(city_model['grid_to_cluster']).fillna(-1).astype(int)
            
            # 处理未知网格（在训练集中未出现的网格）
            unknown_mask = city_data['地理聚类'] == -1
            if unknown_mask.sum() > 0:
                print(f"  {city}有{unknown_mask.sum()}个未知网格样本")
                # 尝试使用区县信息
                unknown_data = city_data[unknown_mask].copy()
                for idx, row in unknown_data.iterrows():
                    # 查找同区县的已知聚类
                    known_in_region = city_data[
                        (city_data['区县'] == row['区县']) & 
                        (city_data['地理聚类'] != -1)
                    ]
                    if len(known_in_region) > 0:
                        city_data.loc[idx, '地理聚类'] = known_in_region['地理聚类'].mode().iloc[0]
            
            # 应用聚类层级
            cluster_sizes = city_model['cluster_sizes']
            
            def assign_cluster_level(cluster_id):
                if cluster_id == -1:
                    return 0
                size = cluster_sizes.get(cluster_id, 0)
                if size >= 100:
                    return 3
                elif size >= 20:
                    return 2
                elif size >= 5:
                    return 1
                else:
                    return 0
            
            city_data['聚类层级'] = city_data['地理聚类'].apply(assign_cluster_level)
            
            results.append(city_data)
        
        # 合并所有城市的结果
        result_df = pd.concat(results, ignore_index=True)
        
        # 删除临时列
        result_df = result_df.drop(['grid_lon', 'grid_lat', 'grid_id'], axis=1)
        
        print(f"测试集聚类完成，总样本数: {len(result_df)}")
        print(f"聚类统计: {result_df['地理聚类'].nunique()} 个聚类")
        print(f"聚类层级分布: {result_df['聚类层级'].value_counts().to_dict()}")
        
        return result_df


def simplified_geo_clustering(train_df, test_df=None, grid_size=0.01, min_samples_per_grid=5):
    """
    简化的地理聚类方法
    
    参数:
        train_df: 训练集数据
        test_df: 测试集数据（可选）
        grid_size: 网格大小
        min_samples_per_grid: 每个网格最少样本数
    
    返回:
        聚类后的数据（包含'地理聚类'和'聚类层级'列）
    """
    clusterer = SimplifiedGeoClusterMapper(
        grid_size=grid_size, 
        min_samples_per_grid=min_samples_per_grid
    )
    
    train_clustered = clusterer.fit(train_df)
    
    if test_df is not None:
        test_clustered = clusterer.transform(test_df)
        return train_clustered, test_clustered
    else:
        return train_clustered


def main():
    # 确保数据包含必要的列
    required_columns = ['城市', '区县', 'lon', 'lat']
    
    # 检查训练集
    missing_cols = [col for col in required_columns if col not in rent_train.columns]
    if missing_cols:
        print(f"训练集缺少列: {missing_cols}")
        return None, None
    
    # 检查测试集
    if rent_test is not None:
        missing_cols = [col for col in required_columns if col not in rent_test.columns]
        if missing_cols:
            print(f"测试集缺少列: {missing_cols}")
            return None, None
    
    # 执行聚类
    train_clustered, test_clustered = simplified_geo_clustering(
        rent_train, rent_test, grid_size=0.01, min_samples_per_grid=5
    )
    
    # 验证结果
    print("\n=== 聚类结果验证 ===")
    print("训练集新增列:", [col for col in train_clustered.columns if col not in rent_train.columns])
    if test_clustered is not None:
        print("测试集新增列:", [col for col in test_clustered.columns if col not in rent_test.columns])
    
    # 显示聚类统计
    print("\n训练集聚类统计:")
    cluster_stats_train = train_clustered.groupby(['城市', '区县', '地理聚类', '聚类层级']).size().reset_index(name='数量')
    print(cluster_stats_train.head(10))
    
    if test_clustered is not None:
        print("\n测试集聚类统计:")
        cluster_stats_test = test_clustered.groupby(['城市', '区县', '地理聚类', '聚类层级']).size().reset_index(name='数量')
        print(cluster_stats_test.head(10))
    
    return train_clustered, test_clustered


if __name__ == "__main__":
    train_result, test_result = main()
    
    # 验证结果是否正确添加到数据中
    if train_result is not None:
        print(f"\n训练集最终形状: {train_result.shape}")
        print(f"包含的列: {list(train_result.columns)}")
        print(f"'地理聚类'列的值范围: {sorted(train_result['地理聚类'].unique())}")
        print(f"'聚类层级'列的值范围: {sorted(train_result['聚类层级'].unique())}")

开始地理聚类...
处理城市: 0, 样本数: 14542
  0有266个稀疏网格样本需要处理
处理城市: 1, 样本数: 7725
  1有36个稀疏网格样本需要处理
处理城市: 2, 样本数: 12059
  2有193个稀疏网格样本需要处理
处理城市: 3, 样本数: 15550
  3有214个稀疏网格样本需要处理
处理城市: 4, 样本数: 11796
  4有321个稀疏网格样本需要处理
处理城市: 5, 样本数: 4077
  5有44个稀疏网格样本需要处理
处理城市: 6, 样本数: 1020
  6有59个稀疏网格样本需要处理
处理城市: 7, 样本数: 10211
  7有29个稀疏网格样本需要处理
处理城市: 8, 样本数: 7274
  8有80个稀疏网格样本需要处理
处理城市: 9, 样本数: 1255
  9有45个稀疏网格样本需要处理
处理城市: 10, 样本数: 12979
  10有142个稀疏网格样本需要处理
处理城市: 11, 样本数: 411
  11有49个稀疏网格样本需要处理
地理聚类完成，总样本数: 98899
聚类统计: 425 个聚类
聚类层级分布: {2: 44632, 3: 44034, 1: 10187, 0: 46}
对测试集应用地理聚类...
  1有13个未知网格样本
  10有39个未知网格样本
  3有95个未知网格样本
  0有76个未知网格样本
  2有28个未知网格样本
  9有32个未知网格样本
  4有56个未知网格样本
  5有29个未知网格样本
  8有29个未知网格样本
  7有6个未知网格样本
  11有20个未知网格样本
  6有10个未知网格样本
测试集聚类完成，总样本数: 9773
聚类统计: 385 个聚类
聚类层级分布: {2: 4543, 3: 3894, 1: 1172, 0: 164}

=== 聚类结果验证 ===
训练集新增列: ['地理聚类', '聚类层级']
测试集新增列: ['地理聚类', '聚类层级']

训练集聚类统计:
   城市   区县  地理聚类  聚类层级  数量
0   0  5.0    49     2  71
1   0  5.0    71     2   9
2   0  5.0    83     2  52
3   0  5.

In [14]:
rent_train=train_result
rent_test=test_result

In [15]:
def safe_float_conversion(value, default=0.0):
    """
    安全地将值转换为浮点数
    """
    try:
        return float(value)
    except (ValueError, TypeError):
        return default

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    计算两个经纬度坐标之间的球面距离（公里）
    """
    # 安全地转换为浮点数
    lat1 = safe_float_conversion(lat1)
    lon1 = safe_float_conversion(lon1)
    lat2 = safe_float_conversion(lat2)
    lon2 = safe_float_conversion(lon2)
    
    # 检查是否有无效的坐标值
    if any(pd.isna([lat1, lon1, lat2, lon2])):
        return 0
    
    R = 6371  # 地球半径，单位：公里
    
    try:
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        
        return R * c
    except Exception as e:
        print(f"距离计算错误: {e}, 坐标: ({lat1}, {lon1}), ({lat2}, {lon2})")
        return 0

def calculate_city_centers(df, city_col='城市', lon_col='lon', lat_col='lat', method='convex_hull'):
    """
    自动计算每个城市的中心点
    """
    city_centers = {}
    
    # 确保我们只使用有效的经纬度列
    required_cols = [city_col, lon_col, lat_col]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"数据中缺少必要的列: {missing_cols}")
    
    # 按城市分组
    grouped = df.groupby(city_col)
    
    for city_id, group in grouped:
        # 确保经纬度是数值类型
        group = group.copy()
        group[lon_col] = group[lon_col].apply(safe_float_conversion)
        group[lat_col] = group[lat_col].apply(safe_float_conversion)
        
        # 移除无效的坐标
        valid_coords = group[(group[lon_col] != 0) & (group[lat_col] != 0)]
        
        if len(valid_coords) < 1:
            print(f"警告: 城市 {city_id} 没有有效的坐标数据")
            continue
            
        if len(valid_coords) < 3:  # 如果数据点太少，使用简单平均
            center_lon = valid_coords[lon_col].mean()
            center_lat = valid_coords[lat_col].mean()
        else:
            if method == 'mean':
                center_lon = valid_coords[lon_col].mean()
                center_lat = valid_coords[lat_col].mean()
            elif method == 'median':
                center_lon = valid_coords[lon_col].median()
                center_lat = valid_coords[lat_col].median()
            elif method == 'convex_hull':
                # 使用凸包中心作为城市中心
                try:
                    points = valid_coords[[lon_col, lat_col]].values
                    hull = ConvexHull(points)
                    # 计算凸包顶点的中心
                    hull_points = points[hull.vertices]
                    center_lon = hull_points[:, 0].mean()
                    center_lat = hull_points[:, 1].mean()
                except Exception as e:
                    print(f"城市 {city_id} 凸包计算失败: {e}，使用中位数代替")
                    center_lon = valid_coords[lon_col].median()
                    center_lat = valid_coords[lat_col].median()
            else:
                raise ValueError(f"不支持的method: {method}")
        
        city_centers[city_id] = (center_lon, center_lat)
    
    print(f"计算了 {len(city_centers)} 个城市的中心点")
    return city_centers

def calculate_ring_thresholds(df, city_col='城市', ring_col='环线位置', 
                             lon_col='lon', lat_col='lat', city_centers=None):
    """
    自动计算每个城市的环线距离阈值
    """
    if city_centers is None:
        city_centers = calculate_city_centers(df, city_col, lon_col, lat_col)
    
    ring_definitions = {}
    
    # 按城市分组
    grouped = df.groupby(city_col)
    
    for city_id, group in grouped:
        if city_id not in city_centers:
            continue
            
        center_lon, center_lat = city_centers[city_id]
        
        # 计算每个点到城市中心的距离
        distances = []
        rings = []
        for _, row in group.iterrows():
            if pd.notna(row[ring_col]) and row[ring_col] != '':
                # 确保经纬度是数值类型
                lat_val = safe_float_conversion(row[lat_col])
                lon_val = safe_float_conversion(row[lon_col])
                
                # 跳过无效坐标
                if lat_val == 0 and lon_val == 0:
                    continue
                    
                distance = haversine_distance(
                    lat_val, lon_val, center_lat, center_lon
                )
                distances.append(distance)
                rings.append(row[ring_col])
        
        if not distances:
            continue
        
        # 创建距离和环线的对应关系
        ring_distance_pairs = list(zip(rings, distances))
        
        # 按环线分组，计算每个环线的平均距离
        ring_stats = {}
        for ring in set(rings):
            ring_distances = [d for r, d in ring_distance_pairs if r == ring]
            if ring_distances:
                ring_stats[ring] = {
                    'mean': np.mean(ring_distances),
                    'median': np.median(ring_distances),
                    'min': np.min(ring_distances),
                    'max': np.max(ring_distances),
                    'count': len(ring_distances)
                }
        
        # 确定环线距离阈值（使用最大值）
        ring_thresholds = {}
        sorted_rings = sorted(ring_stats.keys(), 
                             key=lambda x: ring_stats[x]['mean'])
        
        for i, ring in enumerate(sorted_rings):
            if i < len(sorted_rings) - 1:
                # 当前环线的最大距离作为阈值
                ring_thresholds[ring] = ring_stats[ring]['max']
            else:
                # 最后一个环线使用一个较大的值
                ring_thresholds[ring] = ring_stats[ring]['max'] * 1.5
        
        ring_definitions[city_id] = {
            'center': (center_lon, center_lat),
            'rings': ring_thresholds
        }
    
    print(f"为 {len(ring_definitions)} 个城市计算了环线阈值")
    return ring_definitions


In [16]:
class AutoCenterRingLinePredictor:
    """
    自动计算城市中心的环线预测器
    """
    
    def __init__(self):
        self.model = None
        self.features = []
        self.encoders = {}
        self.scalers = {}
        self.city_centers = None
        self.ring_definitions = None
        self.is_trained = False
    
    def auto_define_city_areas(self, df, city_col='城市', ring_col='环线位置',
                              lon_col='lon', lat_col='lat'):
        """
        自动定义城市中心和环线区县
        """
        print("自动计算城市中心和环线区县...")
        
        # 1. 计算城市中心
        self.city_centers = calculate_city_centers(
            df, city_col, lon_col, lat_col, method='convex_hull'
        )
        
        # 2. 计算环线阈值（需要有环线标签的数据）
        if ring_col in df.columns and df[ring_col].notna().any():
            self.ring_definitions = calculate_ring_thresholds(
                df, city_col, ring_col, lon_col, lat_col, self.city_centers
            )
        else:
            print("警告: 没有环线标签数据，无法自动计算环线阈值")
            self.ring_definitions = {}
            
            # 为每个城市设置默认环线定义
            for city_id, center in self.city_centers.items():
                self.ring_definitions[city_id] = {
                    'center': center,
                    'rings': {
                        '一环': 3,
                        '二环': 6,
                        '三环': 10,
                        '四环': 15,
                        '五环': 20,
                        '五环外': 999
                    }
                }
        
        return self.ring_definitions
    
    def calculate_distance(self, lon1, lat1, lon2, lat2):
        """
        计算两个经纬度坐标之间的距离（公里）
        """
        # 使用统一的haversine_distance函数
        return haversine_distance(lat1, lon1, lat2, lon2)
    
    def _prepare_data(self, df):
        """
        数据预处理：确保数据类型正确并清理无效数据
        """
        df_processed = df.copy()
        
        # 打印列信息以便调试
        print(f"数据列: {list(df_processed.columns)}")
        
        # 确保经纬度是数值类型
        if 'lon' in df_processed.columns:
            df_processed['lon'] = df_processed['lon'].apply(safe_float_conversion)
        else:
            raise ValueError("数据中缺少'lon'列")
            
        if 'lat' in df_processed.columns:
            df_processed['lat'] = df_processed['lat'].apply(safe_float_conversion)
        else:
            raise ValueError("数据中缺少'lat'列")
        
        # 删除包含无效经纬度的行
        invalid_coords = df_processed[(df_processed['lon'] == 0) & (df_processed['lat'] == 0)]
        if len(invalid_coords) > 0:
            print(f"警告: 发现 {len(invalid_coords)} 行无效的经纬度数据，将被删除")
            df_processed = df_processed[~((df_processed['lon'] == 0) & (df_processed['lat'] == 0))]
        
        # 检查并清理城市列
        if '城市' not in df_processed.columns:
            raise ValueError("数据中缺少'城市'列")
        
        # 删除城市为空的记录
        city_missing = df_processed['城市'].isna().sum()
        if city_missing > 0:
            print(f"警告: 发现 {city_missing} 行缺失城市数据，将被删除")
            df_processed = df_processed.dropna(subset=['城市'])
        
        return df_processed
    
    def _calculate_distances_to_center(self, df_processed):
        """
        计算每个点到对应城市中心的距离
        """
        distances = []
        for _, row in df_processed.iterrows():
            try:
                city_id = row['城市']
                if city_id in self.city_centers:
                    center_lon, center_lat = self.city_centers[city_id]
                    distance = self.calculate_distance(
                        row['lon'], row['lat'], center_lon, center_lat
                    )
                else:
                    # 未知城市，使用所有城市的平均距离
                    if self.city_centers:
                        all_distances = []
                        for center_lon, center_lat in self.city_centers.values():
                            dist = self.calculate_distance(
                                row['lon'], row['lat'], center_lon, center_lat
                            )
                            all_distances.append(dist)
                        distance = np.mean(all_distances) if all_distances else 0
                    else:
                        distance = 0
                distances.append(distance)
            except Exception as e:
                print(f"计算距离时出错: {e}")
                distances.append(0)
        
        return distances
    
    def create_features(self, df, is_training=True):
        """
        创建特征矩阵
        """
        # 数据预处理
        df_processed = self._prepare_data(df)
        
        # 1. 基础地理特征
        features = ['lon', 'lat']
        
        # 2. 距城市中心距离
        if self.city_centers is None:
            self.auto_define_city_areas(df_processed)
        
        df_processed['距中心距离'] = self._calculate_distances_to_center(df_processed)
        features.append('距中心距离')
        
        # 3. 标准化数值特征
        numerical_features = ['lon', 'lat', '距中心距离']
        if is_training:
            self.scalers['numerical'] = StandardScaler()
            df_processed[numerical_features] = self.scalers['numerical'].fit_transform(
                df_processed[numerical_features]
            )
        else:
            df_processed[numerical_features] = self.scalers['numerical'].transform(
                df_processed[numerical_features]
            )
        
        self.features = features
        return df_processed, features
    
    def train_with_missing_rings(self, df_train, target_col='环线位置', cv_folds=5):
        """
        在训练集上训练模型，处理训练集中的环线缺失
        """
        print("开始训练环线预测模型（处理训练集环线缺失）...")
        
        # 1. 计算城市中心（使用所有训练数据）
        self.city_centers = self.auto_define_city_areas(df_train)
        
        # 2. 分离有环线和无环线的训练数据
        df_train_with_rings = df_train[df_train[target_col].notna() & (df_train[target_col] != '')].copy()
        df_train_missing_rings = df_train[df_train[target_col].isna() | (df_train[target_col] == '')].copy()
        
        print(f"训练集有环线数据: {len(df_train_with_rings)} 条")
        print(f"训练集缺失环线数据: {len(df_train_missing_rings)} 条")
        
        if len(df_train_with_rings) == 0:
            raise ValueError("训练集中没有可用的环线标签数据")
        
        # 3. 准备有环线数据的特征
        df_train_processed, features = self.create_features(df_train_with_rings, is_training=True)
        
        # 4. 分离特征和目标变量
        X_train = df_train_processed[features].values
        y_train = df_train_processed[target_col].values
        
        print(f"有效训练数据: {X_train.shape[0]}")
        print(f"特征数量: {X_train.shape[1]}")
        print(f"环线类别: {np.unique(y_train)}")
        
        # 5. 训练随机森林模型
        self.model = RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
        
        self.model.fit(X_train, y_train)
        
        # 6. 交叉验证评估
        cv_scores = cross_val_score(self.model, X_train, y_train, cv=min(cv_folds, len(y_train)), scoring='accuracy')
        print(f"交叉验证准确率: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        # 7. 训练集上的表现
        y_pred_train = self.model.predict(X_train)
        train_accuracy = accuracy_score(y_train, y_pred_train)
        print(f"训练集准确率: {train_accuracy:.4f}")
        
        # 8. 特征重要性
        feature_importance = pd.DataFrame({
            'feature': features,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\n特征重要性:")
        print(feature_importance)
        
        self.is_trained = True
        print("模型训练完成!")
        
        return self.model, features
    
    def fill_missing_rings(self, df, target_col='环线位置', return_proba=False):
        """
        填充数据集中的缺失环线
        """
        if not self.is_trained:
            raise ValueError("模型尚未训练，请先调用 train_with_missing_rings() 方法")
        
        print(f"开始填充缺失环线...")
        
        # 1. 准备数据（使用训练时的预处理）
        df_processed, _ = self.create_features(df, is_training=False)
        
        # 2. 提取特征
        X = df_processed[self.features].values
        
        # 3. 预测
        predictions = self.model.predict(X)
        
        # 4. 创建结果DataFrame
        df_result = df.copy()
        
        # 5. 只填充缺失的环线
        mask_missing = df_result[target_col].isna() | (df_result[target_col] == '')
        df_result.loc[mask_missing, target_col] = predictions[mask_missing]
        
        print(f"填充了 {mask_missing.sum()} 条缺失环线记录")
        
        # 6. 预测概率（可选）
        if return_proba:
            probabilities = self.model.predict_proba(X)
            confidence = np.max(probabilities, axis=1)
            df_result['环线预测置信度'] = confidence
            
            # 为每个类别保存概率
            classes = self.model.classes_
            for i, class_name in enumerate(classes):
                df_result[f'{class_name}_概率'] = probabilities[:, i]
        
        return df_result



In [17]:
def complete_ring_filling_workflow(rent_train, rent_test, target_col='环线位置'):
    """
    完整的环线填充工作流程 - 处理训练集和测试集的环线缺失
    """
    rent_train = rent_train.copy()
    rent_test = rent_test.copy()

    print("=== 环线填充工作流程开始 ===")
    print(f"训练集原始大小: {len(rent_train)}")
    print(f"测试集原始大小: {len(rent_test)}")
    
    # 1. 初始化预测器
    predictor = AutoCenterRingLinePredictor()
    
    # 2. 分析数据情况
    train_missing = rent_train[target_col].isna().sum() + (rent_train[target_col] == '').sum()
    test_missing = rent_test[target_col].isna().sum() + (rent_test[target_col] == '').sum()
    
    print(f"训练集环线缺失: {train_missing} 条 ({train_missing/len(rent_train)*100:.1f}%)")
    print(f"测试集环线缺失: {test_missing} 条 ({test_missing/len(rent_test)*100:.1f}%)")
    print(f"训练集有环线数据: {len(rent_train) - train_missing} 条")
    
    # 3. 在训练集上训练模型（只使用有环线的数据）
    model, features = predictor.train_with_missing_rings(rent_train, target_col=target_col)
    
    # 4. 填充训练集的缺失环线
    print("\n=== 填充训练集缺失环线 ===")
    rent_train_filled = predictor.fill_missing_rings(rent_train, target_col=target_col, return_proba=True)
    
    # 5. 填充测试集的缺失环线
    print("\n=== 填充测试集缺失环线 ===")
    rent_test_filled = predictor.fill_missing_rings(rent_test, target_col=target_col, return_proba=True)

    return rent_train_filled, rent_test_filled

In [18]:
rent_train_filled, rent_test_filled=complete_ring_filling_workflow(rent_train, rent_test, target_col='环线位置')

=== 环线填充工作流程开始 ===
训练集原始大小: 98899
测试集原始大小: 9773
训练集环线缺失: 69663 条 (70.4%)
测试集环线缺失: 6962 条 (71.2%)
训练集有环线数据: 29236 条
开始训练环线预测模型（处理训练集环线缺失）...
自动计算城市中心和环线区县...
计算了 12 个城市的中心点
为 5 个城市计算了环线阈值
训练集有环线数据: 29236 条
训练集缺失环线数据: 69663 条
数据列: ['城市', '户型', '装修', '楼层', '面积', '交易时间', '付款方式', '租赁方式', '电梯', '车位', '用水', '用电', '燃气', '采暖', '租期', '配套设施', 'lon', 'lat', '区县', '板块', '环线位置', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司', '绿化率', '容积率', '物业费', '建筑结构', '产权描述', '供水', '供暖', '供电', '燃气费', '供热费', '停车位', '停车费用', '卧室数', '客厅数', '卫生间数', '配套设施数量', '设施是否完善', '小区平均房龄', '小区房龄差异', '政策周期', '地理聚类', '聚类层级']
有效训练数据: 29236
特征数量: 3
环线类别: ['三至四环' '中环至外环' '二环内' '二至三环' '五至六环' '六环外' '内环内' '内环至中环' '内环至外环' '四至五环'
 '外环外']
交叉验证准确率: 0.9791 (+/- 0.0615)
训练集准确率: 0.9968

特征重要性:
  feature  importance
1     lat    0.409566
0     lon    0.300432
2   距中心距离    0.290001
模型训练完成!

=== 填充训练集缺失环线 ===
开始填充缺失环线...
数据列: ['城市', '户型', '装修', '楼层', '面积', '交易时间', '付款方式', '租赁方式', '电梯', '车位', '用水', '用电', '燃气', '采暖', '租期', '配套设施', 'lon', 'lat', '区县', '板块', '

In [19]:
columns_to_keep = [col for col in rent_train.columns if '置信度' not in col and '概率' not in col]
rent_train=rent_train_filled[columns_to_keep]

In [20]:
columns_to_keep = [col for col in rent_test.columns if '置信度' not in col and '概率' not in col]
rent_test=rent_test_filled[columns_to_keep]

In [21]:
class DeveloperPropertyProcessor:
    def __init__(self, min_count=30, n_components=5, n_clusters=10, target_col="房屋总数", random_state=111):
        """
        参数:
            min_count: 高频类最小出现次数
            n_components: PCA降维维度
            n_clusters: 聚类数量
            target_col: 用于统计特征的目标列
        """
        self.min_count = min_count
        self.n_components = n_components
        self.n_clusters = n_clusters
        self.target_col = target_col
        self.random_state = random_state
        
        # 存储模型和映射
        self.dev_highfreq = None
        self.prop_highfreq = None
        self.tfidf_dev = None
        self.tfidf_prop = None
        self.pca_dev = None
        self.pca_prop = None
        self.kmeans_dev = None
        self.kmeans_prop = None
        self.dev_stats = None
        self.prop_stats = None

    def clean_name(self, name):
        """文本清洗"""
        unknown_patterns = [
            "无开发商", "无", "暂无", "未知", "未公布", "待定", 
            "自建", "个人", "暂无信息", "未知开发商", "开发商待定",
            "无主", "不详", "不清楚", "未明确", "待确认"
        ]
        if pd.isna(name) or name in unknown_patterns:
            return "未知"
        name = str(name)
        name = re.sub(r"[（）()]", "", name)
        name = re.sub(r"(股份|有限|责任|集团|公司|房地产|开发|物业|管理|服务|控股|建设|实业|投资|发展|地产)", "", name)
        return name.strip().lower()

    def fit(self, df):
        """在训练集上拟合"""
        df["开发商_清洁"] = df["开发商"].apply(self.clean_name)
        df["物业公司_清洁"] = df["物业公司"].apply(self.clean_name)

        #    
        print("开发商清洁后唯一值数量:", df["开发商_清洁"].nunique())
        print("开发商清洁后样本:", df["开发商_清洁"].value_counts().head())

        # 高频类统计
        dev_counts = df["开发商_清洁"].value_counts()
        prop_counts = df["物业公司_清洁"].value_counts()
        self.dev_highfreq = set(dev_counts[dev_counts > self.min_count].index)
        self.prop_highfreq = set(prop_counts[prop_counts > self.min_count].index)

        # 统计特征
        if self.target_col in df.columns:
            self.dev_stats = df.groupby("开发商_清洁")[self.target_col].agg(['mean', 'median', 'std', 'count'])
            self.prop_stats = df.groupby("物业公司_清洁")[self.target_col].agg(['mean', 'median', 'std', 'count'])

        # TF-IDF + PCA
        self.tfidf_dev = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
        self.tfidf_prop = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
        X_dev = self.tfidf_dev.fit_transform(df["开发商_清洁"])
        X_prop = self.tfidf_prop.fit_transform(df["物业公司_清洁"])
        self.pca_dev = PCA(n_components=self.n_components, random_state=self.random_state)
        self.pca_prop = PCA(n_components=self.n_components, random_state=self.random_state)
        X_dev_pca = self.pca_dev.fit_transform(X_dev.toarray())
        X_prop_pca = self.pca_prop.fit_transform(X_prop.toarray())

        # 聚类
        self.kmeans_dev = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_prop = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_dev.fit(X_dev_pca)
        self.kmeans_prop.fit(X_prop_pca)
        return self

    def transform(self, df):
        """在训练集或测试集上应用已拟合的映射"""
        df = df.copy()
        df["开发商_清洁"] = df["开发商"].apply(self.clean_name)
        df["物业公司_清洁"] = df["物业公司"].apply(self.clean_name)

        # TF-IDF + PCA + 聚类
        X_dev = self.tfidf_dev.transform(df["开发商_清洁"])
        X_prop = self.tfidf_prop.transform(df["物业公司_清洁"])
        dev_pca = self.pca_dev.transform(X_dev.toarray())
        prop_pca = self.pca_prop.transform(X_prop.toarray())

        # 聚类编号特征
        dev_clusters = self.kmeans_dev.predict(dev_pca)
        prop_clusters = self.kmeans_prop.predict(prop_pca)
        df["开发商_cluster"] = dev_clusters
        df["物业公司_cluster"] = prop_clusters

        # 用聚类结果替换原始列
        df["开发商"] = dev_clusters
        df["物业公司"] = prop_clusters

        # 删除所有中间列
        columns_to_drop = ['开发商_清洁', '物业公司_清洁','开发商_cluster','物业公司_cluster']
        df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
        return df


In [22]:
class DeveloperPropertyProcessor:
    def __init__(self, min_count=30, n_components=5, n_clusters=10, target_col="房屋总数", 
                 max_features=500, random_state=111):  # 新增 max_features 参数
        """
        参数:
            min_count: 高频类最小出现次数
            n_components: PCA降维维度
            n_clusters: 聚类数量
            target_col: 用于统计特征的目标列
            max_features: TF-IDF最大特征数，防止内存溢出
        """
        self.min_count = min_count
        self.n_components = n_components
        self.n_clusters = n_clusters
        self.target_col = target_col
        self.max_features = max_features  # 限制特征数量
        self.random_state = random_state
        
        # 存储模型和映射
        self.dev_highfreq = None
        self.prop_highfreq = None
        self.tfidf_dev = None
        self.tfidf_prop = None
        self.pca_dev = None
        self.pca_prop = None
        self.kmeans_dev = None
        self.kmeans_prop = None
        self.dev_stats = None
        self.prop_stats = None

    def clean_name(self, name):
        """文本清洗 - 优化版本"""
        if pd.isna(name):
            return "未知"
        
        name = str(name).strip()
        
        # 更全面的未知模式匹配
        unknown_patterns = [
            "无开发商", "无", "暂无", "未知", "未公布", "待定", 
            "自建", "个人", "暂无信息", "未知开发商", "开发商待定",
            "无主", "不详", "不清楚", "未明确", "待确认", "nan", "None"
        ]
        
        if any(pattern in name for pattern in unknown_patterns) or len(name) <= 1:
            return "未知"
        
        # 更温和的文本清理
        name = re.sub(r"[（）()【】\[\]]", "", name)
        # 保留一些关键词，避免过度清理
        name = re.sub(r"\s+", " ", name)
        return name.strip().lower()

    def fit(self, df):
        """在训练集上拟合 - 内存优化版本"""
        # 创建副本避免 SettingWithCopyWarning
        df = df.copy()
        
        df["开发商_清洁"] = df["开发商"].apply(self.clean_name)
        df["物业公司_清洁"] = df["物业公司"].apply(self.clean_name)

        print("开发商清洁后唯一值数量:", df["开发商_清洁"].nunique())
        print("开发商清洁后样本:", df["开发商_清洁"].value_counts().head())

        # 高频类统计
        dev_counts = df["开发商_清洁"].value_counts()
        prop_counts = df["物业公司_清洁"].value_counts()
        self.dev_highfreq = set(dev_counts[dev_counts > self.min_count].index)
        self.prop_highfreq = set(prop_counts[prop_counts > self.min_count].index)

        # 统计特征
        if self.target_col in df.columns:
            self.dev_stats = df.groupby("开发商_清洁")[self.target_col].agg(['mean', 'median', 'std', 'count'])
            self.prop_stats = df.groupby("物业公司_清洁")[self.target_col].agg(['mean', 'median', 'std', 'count'])

        # TF-IDF + PCA - 内存优化
        print("开始TF-IDF转换...")
        
        # 限制特征数量，使用更小的min_df
        self.tfidf_dev = TfidfVectorizer(
            ngram_range=(1, 1),  # 只使用1-gram，减少特征
            min_df=5,           # 增加min_df，减少特征
            max_features=self.max_features,  # 限制最大特征数
            lowercase=False     # 已经转换为小写
        )
        
        self.tfidf_prop = TfidfVectorizer(
            ngram_range=(1, 1),
            min_df=5,
            max_features=self.max_features,
            lowercase=False
        )
        
        # 分批处理或使用稀疏矩阵
        X_dev = self.tfidf_dev.fit_transform(df["开发商_清洁"])
        X_prop = self.tfidf_prop.fit_transform(df["物业公司_清洁"])
        
        print(f"TF-IDF特征维度 - 开发商: {X_dev.shape}, 物业: {X_prop.shape}")
        
        # 使用增量PCA或普通PCA
        from sklearn.decomposition import IncrementalPCA
        
        # 对于大数据集，使用IncrementalPCA
        self.pca_dev = IncrementalPCA(n_components=self.n_components)
        self.pca_prop = IncrementalPCA(n_components=self.n_components)
        
        # 分批处理PCA拟合
        batch_size = 10000
        X_dev_dense = X_dev.toarray() if X_dev.shape[1] < 5000 else X_dev  # 如果特征太多，保持稀疏
        
        if hasattr(X_dev_dense, 'toarray'):
            # 如果是稀疏矩阵且需要分批处理
            X_dev_pca = self._batch_pca_fit_transform(self.pca_dev, X_dev_dense, batch_size)
            X_prop_pca = self._batch_pca_fit_transform(self.pca_prop, X_prop, batch_size)
        else:
            X_dev_pca = self.pca_dev.fit_transform(X_dev_dense)
            X_prop_pca = self.pca_prop.fit_transform(X_prop.toarray())
        
        # 聚类
        print("开始聚类...")
        self.kmeans_dev = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.kmeans_prop = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        
        self.kmeans_dev.fit(X_dev_pca)
        self.kmeans_prop.fit(X_prop_pca)
        
        return self

    def _batch_pca_fit_transform(self, pca, sparse_matrix, batch_size):
        """分批处理PCA拟合和转换"""
        n_samples = sparse_matrix.shape[0]
        result = []
        
        # 分批拟合
        for i in range(0, n_samples, batch_size):
            batch = sparse_matrix[i:i+batch_size].toarray()
            if i == 0:
                pca.fit(batch)
            else:
                pca.partial_fit(batch)
        
        # 分批转换
        for i in range(0, n_samples, batch_size):
            batch = sparse_matrix[i:i+batch_size].toarray()
            transformed = pca.transform(batch)
            result.append(transformed)
        
        return np.vstack(result)

    def transform(self, df):
        """在训练集或测试集上应用已拟合的映射"""
        # 创建副本
        df = df.copy()
        
        df["开发商_清洁"] = df["开发商"].apply(self.clean_name)
        df["物业公司_清洁"] = df["物业公司"].apply(self.clean_name)

        # TF-IDF + PCA + 聚类
        X_dev = self.tfidf_dev.transform(df["开发商_清洁"])
        X_prop = self.tfidf_prop.transform(df["物业公司_清洁"])
        
        # 分批处理转换
        dev_pca = self._batch_pca_transform(self.pca_dev, X_dev)
        prop_pca = self._batch_pca_transform(self.pca_prop, X_prop)

        # 聚类编号特征
        dev_clusters = self.kmeans_dev.predict(dev_pca)
        prop_clusters = self.kmeans_prop.predict(prop_pca)
        
        # 直接替换原始列
        df["开发商"] = dev_clusters
        df["物业公司"] = prop_clusters

        # 删除中间列
        columns_to_drop = ['开发商_清洁', '物业公司_清洁']
        df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
        
        return df

    def _batch_pca_transform(self, pca, sparse_matrix, batch_size=10000):
        """分批处理PCA转换"""
        n_samples = sparse_matrix.shape[0]
        result = []
        
        for i in range(0, n_samples, batch_size):
            batch = sparse_matrix[i:i+batch_size].toarray()
            transformed = pca.transform(batch)
            result.append(transformed)
        
        return np.vstack(result)

In [23]:
processor = DeveloperPropertyProcessor(
    min_count=30,
    n_components=3,
    n_clusters=10,
    target_col="房屋总数"
)

# === 训练阶段 ===
processor.fit(rent_train)
rent_train = processor.transform(rent_train)

# === 测试集映射 ===
rent_test = processor.transform(rent_test)

开发商清洁后唯一值数量: 2069
开发商清洁后样本: 开发商_清洁
未知                 31778
三河市莲荷房地产开发有限公司      1523
三河顺通房地产开发有限公司        834
广州城建开发南沙房地产有限公司      616
昆明宝华房地产开发有限公司        534
Name: count, dtype: int64
开始TF-IDF转换...
TF-IDF特征维度 - 开发商: (98899, 500), 物业: (98899, 500)
开始聚类...


In [24]:
def simplify_lease_period(df, column_name='租期'):
    """
    简化租期分类
    """
    def process_single_lease(text):
        if pd.isna(text) or text == '':
            return '未知'
        
        text = str(text).strip()
        
        # 提取数字
        numbers = re.findall(r'\d+', text)
        if not numbers:
            return '未知'
        
        # 转换为月份数
        if '年' in text:
            # 如果有"年"字，转换为月份
            if '以内' in text or '以上' in text:
                # 处理"X年以内/以上"的情况
                years = int(numbers[0])
                if '以内' in text:
                    return '长期' if years >= 2 else '中期'
                else:  # 以上
                    return '长期'
            elif '~' in text or '~' in text:
                # 处理范围
                if len(numbers) >= 2:
                    min_months = int(numbers[0]) * 12
                    max_months = int(numbers[1]) * 12
                    avg_months = (min_months + max_months) / 2
                else:
                    avg_months = int(numbers[0]) * 12
            else:
                # 固定年数
                avg_months = int(numbers[0]) * 12
        else:
            # 只有月份
            if '以内' in text or '以上' in text:
                # 处理"X个月以内/以上"的情况
                months = int(numbers[0])
                if '以内' in text:
                    return '短期' if months <= 3 else '中期'
                else:  # 以上
                    return '长期' if months >= 12 else '中期'
            elif '~' in text or '~' in text:
                # 处理范围
                if len(numbers) >= 2:
                    avg_months = (int(numbers[0]) + int(numbers[1])) / 2
                else:
                    avg_months = int(numbers[0])
            else:
                # 固定月数
                avg_months = int(numbers[0])
        
        # 根据平均月份数分类
        if 'avg_months' in locals():
            if avg_months < 6:
                return '短期'
            elif avg_months < 12:
                return '中期'
            else:
                return '长期'
        
        return '未知'
    
    # 应用处理函数到指定列
    df['租期'] = df[column_name].apply(process_single_lease)
    
    # 可选：创建租期分类的二进制列
    lease_categories = ['短期', '中期', '长期']
    for category in lease_categories:
        df[f'租期_{category}'] = df['租期'].apply(
            lambda x: 1 if x == category else 0
        )
    
    return df


rent_train= simplify_lease_period(rent_train)
rent_test= simplify_lease_period(rent_test)

In [25]:
def main_property_type(df):
    # 定义主要产权描述
    main_property_types = {
        '商品房': '商品房',
        '已购公房': '公房',
        '经济适用房': '经适房',
        '央产房': '公房',
        '房改房': '公房',
        '限价商品房': '限价房',
        '自住型商品房': '自住型商品房',
        '二类经济适用房': '经适房',
        '一类经济适用房': '经适房',
        '使用权': '使用权',
        '私产': '私产',
        '公租房': '公租房',
        '定向安置房': '安置房',
        '动迁安置房': '安置房',
        '拆迁还建房': '安置房'
    }
    
    def extract_main_property_type(text):
        if pd.isna(text) or text == '':
            return '其他'
        
        text = str(text)
        # 按优先级查找主要产权描述
        for key, value in main_property_types.items():
            if key in text:
                return value
        
        return '其他'
    
    df['产权描述'] = df['产权描述'].apply(extract_main_property_type)
    
    # 可选：创建产权描述是否为商品房的二进制列
    df['是否商品房'] = df['产权描述'].apply(
        lambda x: 1 if not pd.isna(x) and '商品房' in str(x) else 0
    )

    return df

rent_train=main_property_type(rent_train)
rent_test=main_property_type(rent_test)

In [26]:
def partly_overlap(df):
    if '用水' in df.columns and '供水' in df.columns:
        df['用水_合并'] = df['用水'].copy()
        
        # 找到'用水'列为空但'供水'列不为空的位置
        mask = df['用水_合并'].isna() & df['供水'].notna()
        
        # 用'供水'列的值填补空白
        df.loc[mask, '用水_合并'] = df.loc[mask, '供水']

        df['用水']=df['用水_合并']
        
    if '用电' in df.columns and '供电' in df.columns:
        df['用电_合并'] = df['用电'].copy()
        
        # 找到'用电'列为空但'供电'列不为空的位置
        mask = df['用电_合并'].isna() & df['供电'].notna()
        
        # 用'供电'列的值填补空白
        df.loc[mask, '用电_合并'] = df.loc[mask, '供电']

        df['用电']=df['用电_合并']

    if '采暖' in df.columns and '供暖' in df.columns:
        df['采暖_合并'] = df['采暖'].copy()
        
        # 找到'采暖'列为空但'供暖'列不为空的位置
        mask = df['采暖_合并'].isna() & df['供暖'].notna()
        
        # 用'供暖'列的值填补空白
        df.loc[mask, '采暖_合并'] = df.loc[mask, '供暖']

        df['采暖']=df['采暖_合并']


    return df


rent_train=partly_overlap(rent_train)
rent_test=partly_overlap(rent_test)


In [27]:
def convert_object(train_df, test_df, columns):
    #装修情况、建筑结构、交易权属、房屋用途、房屋年限、建筑结构、供水、供暖
    missing_fill_map = {
        '环线位置':'未知',
        '付款方式':'未知',
        '车位':'未知',
        '建筑结构': '未知', 
        '产权描述':'其他',
        '用水_合并': '未知',
        '用电_合并': '未知',
        '采暖_合并': '未知',
        '租期':'未知',
        '政策周期':'未知',
    }
    for col in columns:
        if col not in train_df.columns:
            continue
        fill_value = missing_fill_map.get(col, '未知')
        train_df[col] = train_df[col].fillna(fill_value)
        test_df[col] = test_df[col].fillna(fill_value)
        if col == '环线位置':
            category_order = [['三至四环','中环至外环','二环内','二至三环','五至六环','六环外','内环内','内环至中环','内环至外环','四至五环'
,'外环外']]
            # 创建编码器，处理未知类别（设为-1）
            encoder = OrdinalEncoder(
                categories=category_order,
                handle_unknown='use_encoded_value',
                unknown_value=-1  # 将未知类别编码为-1
            )
            
            # 对训练集进行拟合和转换
            train_df['环线位置'] = encoder.fit_transform(train_df[['环线位置']])
            # 对测试集进行转换
            test_df['环线位置'] = encoder.transform(test_df[['环线位置']])
            
        else:
            le = LabelEncoder()
            le.fit(train_df[col].fillna("__NA__"))
            mapping = dict(zip(le.classes_, le.transform(le.classes_)))
            train_df[col] = train_df[col].fillna("__NA_").map(mapping)
            test_df[col] = test_df[col].fillna("__NA__").map(lambda x: mapping.get(x, -1))
        
    return train_df, test_df

columns=['环线位置','付款方式','车位','建筑结构','产权描述','用水_合并','用电_合并','采暖_合并','租期','政策周期']
rent_train, rent_test=convert_object(rent_train, rent_test, columns)

In [28]:
knn_cols = ['停车费用','小区平均房龄','供热费','停车位','物业费','卧室数','卫生间数']
knn_cols = [c for c in knn_cols if c in rent_train.columns]
date_cols = ['交易时间']
date_cols = [c for c in date_cols if c in rent_train.columns]

if knn_cols:
    print(f"🔍 使用 sklearn KNNImputer 插值列: {knn_cols}")
    
    # 使用sklearn的KNNImputer（更节省内存）
    knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')
    train_knn = knn_imputer.fit_transform(rent_train[knn_cols])
    test_knn = knn_imputer.transform(rent_test[knn_cols])
    
    rent_train[knn_cols] = train_knn
    rent_test[knn_cols] = test_knn

# 处理日期列
for col in date_cols:
    missing_count = rent_train[col].isna().sum()
    print(f"  {col}: 训练集缺失值 {missing_count}/{len(rent_train)}")
    missing_count_test = rent_test[col].isna().sum()
    print(f"  {col}: 测试集缺失值 {missing_count_test}/{len(rent_test)}")

# 其他数值列 → 中位数填充

remaining_num_cols = [col for col in num_cols if col not in knn_cols and col not in date_cols]

for col in remaining_num_cols:
    med = rent_train[col].median()  # 注意：这里应该是rent_train而不是train
    rent_train[col] = rent_train[col].fillna(med)
    rent_test[col] = rent_test[col].fillna(med)

🔍 使用 sklearn KNNImputer 插值列: ['停车费用', '小区平均房龄', '供热费', '停车位', '物业费', '卧室数', '卫生间数']
  交易时间: 训练集缺失值 0/98899
  交易时间: 测试集缺失值 0/9773


In [29]:
def house_evaluation(df,bed_weight=0.6,liv_weight=0.15,bath_weight=0.15):
    df['户型评分']=df['卧室数']*bed_weight+df['客厅数']*liv_weight+df['卫生间数']*bath_weight
    return df

rent_train= house_evaluation(rent_train)
rent_test=house_evaluation(rent_test)

In [31]:
# 添加Price列回训练集
rent_train['Price'] = target_Price.values

# 添加ID列回测试集  
rent_test['ID'] = rent_test_id.values

# 保存
rent_train.to_csv('rent_train_processed.csv', index=False)
rent_test.to_csv('rent_test_processed.csv', index=False)

print("✓ 数据已保存")

✓ 数据已保存
